# Tesseract on Colab — Arabic OCR benchmark

Measures `tesseract-best` (the `tessdata_best` Arabic model) on your own PDF:
seconds per page, thread scaling, and — if you supply ground truth — CER and WER.

**Why this notebook exists.** The deployment box is a `t3.medium` whose cores are
about **3.4× slower** than a development laptop (a fixed 8M-iteration Python loop:
1.48 s there, 0.44 s here). Quoting a laptop's seconds-per-page at a server is how
a 214-page book came to need 1136 s against a 540 s budget. Run this on Colab to
get a *third* reference point, and read the ratio rather than the absolute number.

CPU only — Tesseract does not use a GPU. **Runtime → Change runtime type → CPU**
is fine and starts faster.

Pairs with `qari_benchmark.ipynb`, which scores the same pages the same way.

## 1. Install

In [ ]:
# tessdata_best is a *different model* from the `ara` pack distributions ship,
# not a speed setting on the same one: 0.172 WER against 0.545 measured on this
# project's fixtures. Benchmarking "Tesseract" on the fast model is how it earns
# its reputation for being hopeless at Arabic.
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-ara > /dev/null
# No --upgrade and nothing Colab preloads: this notebook needs no restart.
# pillow is deliberately untouched -- Colab imports PIL at startup, and
# replacing it under a live kernel leaves a mixed install that fails later
# as `cannot import name '_Ink' from 'PIL._typing'`.
!pip -q install pytesseract pymupdf

import os, urllib.request
BEST = "/usr/share/tessdata-best"
os.makedirs(BEST, exist_ok=True)
url = "https://github.com/tesseract-ocr/tessdata_best/raw/main/ara.traineddata"
if not os.path.exists(f"{BEST}/ara.traineddata"):
    urllib.request.urlretrieve(url, f"{BEST}/ara.traineddata")

!tesseract --version | head -2
print("models:", os.listdir(BEST))
print("cpus:", os.cpu_count())

## 2. The PDF

In [ ]:
# --- get a PDF in ---------------------------------------------------------
# Drag a file into the file browser on the left and set PDF_PATH, or run this
# and pick one. For anything large, mount Drive instead -- an upload widget on
# a 25 MB book is slower than Drive and dies on a flaky connection.
from google.colab import files
import os

PDF_PATH = "/content/book.pdf"

if not os.path.exists(PDF_PATH):
    uploaded = files.upload()
    name = next(iter(uploaded))
    os.rename(name, PDF_PATH)

import pymupdf
doc = pymupdf.open(PDF_PATH)
print(f"{doc.page_count} pages")

# Which pages to test. A handful is enough: these engines are seconds per page
# and the numbers stabilise quickly. Pick from the middle -- front matter and
# title pages are not representative of body text.
FIRST, LAST = 40, 47
PAGES = list(range(FIRST, min(LAST + 1, doc.page_count)))
print("testing pages", PAGES)


## 3. Scoring

In [ ]:
# --- scoring -------------------------------------------------------------
# Both notebooks score the same way so their numbers can sit in one table.
# CER and WER are both reported because on fragmented Arabic they disagree:
# splitting `اليسار` into `ا ليسا ر` changes no letters and destroys every
# word, so CER barely moves while WER collapses -- and WER is the one that
# predicts whether retrieval works.
import unicodedata, re

def normalize(text: str) -> str:
    """NFKC, strip bidi controls, collapse whitespace.

    Presentation forms (U+FB50-FDFF, U+FE70-FEFF) fold to typed letters here,
    so an engine is not punished for emitting a form the pipeline normalises
    away before indexing anyway.
    """
    text = unicodedata.normalize("NFKC", text)
    text = text.translate(dict.fromkeys(map(ord, "\u200e\u200f\u202a\u202b\u202c\u202d\u202e\u2066\u2067\u2068\u2069")))
    return re.sub(r"\s+", " ", text).strip()

def levenshtein(a, b):
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def cer(truth, hyp):
    t, h = normalize(truth), normalize(hyp)
    return 1.0 if not t else min(1.0, levenshtein(t, h) / len(t))

def wer(truth, hyp):
    t, h = normalize(truth).split(), normalize(hyp).split()
    return 1.0 if not t else min(1.0, levenshtein(t, h) / len(t))

def space_ratio(text):
    """Intrinsic quality signal for when there is no ground truth.

    Healthy Arabic prose sits around 0.13-0.22. Far below means words fused
    together; far above means they were split mid-word, which is the failure
    this whole exercise is about.
    """
    t = normalize(text)
    return 0.0 if not t else t.count(" ") / len(t)


## 4. Read the pages

`--psm 6` treats the page as one uniform block, which suits body text; the
default tries to detect layout and on dense right-to-left prose invents columns
that are not there.

`OMP_THREAD_LIMIT=1` matters more than it looks. Tesseract is built with OpenMP
and starts its own threads per page; stacked under a page-level pool that
oversubscribes every core and runs *slower* than either level of parallelism
alone. One page per thread, one thread per page.

In [ ]:
import os, time, io
os.environ["OMP_THREAD_LIMIT"] = "1"
import pytesseract, pymupdf
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

DPI = 300   # the floor most engines are trained around; below it Arabic
            # diacritics and the dots separating ب ت ث merge into one pixel

def render(page_no, dpi=DPI):
    d = pymupdf.open(PDF_PATH)
    try:
        pix = d[page_no].get_pixmap(dpi=dpi)
        return Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
    finally:
        d.close()

def read(page_no, dpi=DPI):
    return pytesseract.image_to_string(
        render(page_no, dpi), lang="ara",
        config=f"--psm 6 --tessdata-dir {BEST}",
    )

t0 = time.perf_counter()
texts = {n: read(n) for n in PAGES}
serial = time.perf_counter() - t0

print(f"serial: {serial/len(PAGES):.2f} s/page over {len(PAGES)} pages")
for n in PAGES[:2]:
    print(f"\n--- page {n} (space ratio {space_ratio(texts[n]):.3f}) ---")
    print(texts[n][:300])

## 5. Thread scaling

Threads rather than processes, deliberately: `pytesseract` runs the `tesseract`
binary as a subprocess, so the GIL is released for the whole of the work. It is
also the only option that works inside a Celery prefork worker, which is
daemonic and may not fork.

Expect the speedup to fall well short of the thread count once you exceed the
real core count — and on a busy 2-core box, to be close to nothing.

In [ ]:
import multiprocessing
cores = os.cpu_count() or 2
counts = sorted({1, 2, 4, cores})

print(f"{'threads':>8}  {'s/page':>7}  {'speedup':>7}")
base = None
for w in counts:
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=w) as pool:
        list(pool.map(read, PAGES))
    per = (time.perf_counter() - t0) / len(PAGES)
    base = base or per
    print(f"{w:>8}  {per:>7.2f}  {base/per:>6.2f}x")

print("\nEach concurrent page also holds a 300-dpi RGB raster, ~26 MB for A4 —")
print("so this is the memory knob as much as the speed one.")

## 6. DPI: measured, not assumed

Lowering DPI looks like free speed and is not. Rendering is a tenth of a second;
the cost is inside Tesseract's LSTM over text lines, which does not shrink
proportionally. Check what you would actually pay in accuracy before trading.

In [ ]:
ref = {n: read(n, 300) for n in PAGES[:4]}

print(f"{'dpi':>5}  {'s/page':>7}  {'agreement vs 300dpi':>20}")
for dpi in (300, 200, 150):
    t0 = time.perf_counter()
    got = {n: read(n, dpi) for n in PAGES[:4]}
    per = (time.perf_counter() - t0) / 4
    agree = sum(
        1 - min(1.0, levenshtein(normalize(ref[n]), normalize(got[n])) / max(1, len(normalize(ref[n]))))
        for n in PAGES[:4]
    ) / 4
    print(f"{dpi:>5}  {per:>7.2f}  {agree:>19.3f}")

## 7. Ground truth (optional)

Everything above is speed and intrinsic signal. For CER and WER you need the
correct text. Type or paste it for one page — one careful page is worth more
than a guess about ten.

In [ ]:
TRUTH = {
    # 40: "اليسار حينئذ بديدو ومعناه الهاربة ...",
}

if not TRUTH:
    print("No ground truth supplied — reporting the intrinsic signal instead.")
    print(f"{'page':>5}  {'chars':>6}  {'space ratio':>12}  verdict")
    for n, t in texts.items():
        r = space_ratio(t)
        verdict = "looks healthy" if 0.13 <= r <= 0.22 else ("fused?" if r < 0.13 else "fragmented?")
        print(f"{n:>5}  {len(normalize(t)):>6}  {r:>12.3f}  {verdict}")
else:
    print(f"{'page':>5}  {'CER':>6}  {'WER':>6}")
    for n, truth in TRUTH.items():
        print(f"{n:>5}  {cer(truth, texts[n]):>6.3f}  {wer(truth, texts[n]):>6.3f}")
    print("\nCER and WER diverging is the expected shape on fragmented Arabic:")
    print("every character survives while no searchable word does.")